# wordle-0.2b — play & evaluate

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch

from wordle02b import Vocabulary, load_word_lists, build_pattern_matrix_cached

vocab = Vocabulary(*load_word_lists())
P = build_pattern_matrix_cached(vocab, cache_dir="data/cache")

# point at your checkpoint; falls back to the most recent one in checkpoints/
CKPT = "checkpoints/wordle-0.2b-final.pt"
if not Path(CKPT).exists():
    cands = sorted(Path("checkpoints").glob("*.pt")) if Path("checkpoints").exists() else []
    CKPT = str(cands[-1]) if cands else None
print("checkpoint:", CKPT)

from wordle02b.model import GPT
device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
model = GPT.load(CKPT, device=device) if CKPT else None
print("model loaded" if model else "no checkpoint found -> baseline only")

## Model vs teacher

Same secret words (drawn from the 200-answer holdout that training never saw), same rules. The teacher is the entropy solver that generated the training data. Once fully trained, the 0.2B model should land close to it (95%+ solve, ~3.8 avg). A partially trained model will lag, and that gap is your training progress bar. During evaluation the model is not allowed guesses that contradict known feedback — such guesses can never be correct, so the filter only removes bad moves.

In [ ]:
from wordle02b.evaluate import play_games, play_games_baseline, split_answers, summarize

N = 300
_, holdout_ids = split_answers(vocab, holdout_size=200, seed=42)
secrets = [vocab.word(int(wid)) for wid in np.random.default_rng(100).choice(holdout_ids, N, replace=False)]

st_model = summarize(play_games(model, vocab, N, secrets, device=device)) if model else None
st_base = summarize(play_games_baseline(vocab, P, N, secrets, seed=100))

for name, st in [("model", st_model), ("teacher", st_base)]:
    if st:
        print(f"{name:8s} solve {st['solve_rate']:.3f}  avg {st['avg_guesses']:.2f}  lost {st['lost']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

labels = ["model", "teacher"] if st_model else ["teacher"]
vals = [st_model["solve_rate"] * 100 if st_model else 0, st_base["solve_rate"] * 100]
axes[0].bar(labels, vals, color=["#6aaa64", "#787c7e"])
axes[0].set_ylim(0, 105); axes[0].set_title("solve rate %"); axes[0].set_ylabel("%")

for i, v in enumerate(vals):
    axes[0].text(i, v + 1, f"{v:.1f}", ha="center")

dist_model = st_model["guess_distribution"] if st_model else {}
dist_base = st_base["guess_distribution"]
x = np.arange(1, 7)
axes[1].bar(x - 0.18, [dist_model.get(k, 0) for k in x], 0.36, label="model", color="#6aaa64")
axes[1].bar(x + 0.18, [dist_base.get(k, 0) for k in x], 0.36, label="teacher", color="#787c7e")
axes[1].set_xlabel("guesses to solve"); axes[1].set_ylabel("games")
axes[1].set_title("guess distribution"); axes[1].legend()
plt.tight_layout()
Path("assets").mkdir(exist_ok=True)
fig.savefig("assets/solve_rate_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Play against it

Type a word, hit Guess. The Hint button asks the model what it would play; Watch model play runs the game to the end.

In [ ]:
import ipywidgets as w
from IPython.display import display

from wordle02b.game import WordleGame, feedback

class PlayUI:
    def __init__(self, model, vocab):
        self.model, self.vocab = model, vocab
        self.reset()
        self.board = w.HTML(); self.status = w.HTML("new game")
        self.guess_box = w.Text(placeholder="5-letter word")
        self.go = w.Button(description="Guess", button_style="success")
        self.hint = w.Button(description="Hint")
        self.autoplay = w.Button(description="Watch model play")
        self.new = w.Button(description="New game")
        self.go.on_click(self.do_guess); self.hint.on_click(self.do_hint)
        self.autoplay.on_click(self.do_autoplay); self.new.on_click(lambda _: self.reset())
        self.render()

    def reset(self):
        self.secret = str(np.random.default_rng().choice(self.vocab.answers))
        self.guesses, self.feedbacks = [], []

    def tile(self, letter, color):
        bg = {"G": "#6aaa64", "Y": "#c9b458", "B": "#787c7e"}[color]
        return (f'<div style="width:44px;height:44px;background:{bg};color:#fff;'
                f'font:bold 22px sans-serif;display:inline-flex;align-items:center;'
                f'justify-content:center;margin:2px;border-radius:4px">{letter}</div>')

    def render(self):
        rows = []
        for i in range(6):
            if i < len(self.guesses):
                cells = "".join(self.tile(l.upper(), c) for l, c in zip(self.guesses[i], self.feedbacks[i]))
            else:
                cells = "".join('<div style="width:44px;height:44px;background:#3a3a3c;display:inline-flex;margin:2px;border-radius:4px"></div>' for _ in range(5))
            rows.append(f"<div>{cells}</div>")
        self.board.value = (f'<div style="font-family:sans-serif;background:#121213;padding:10px;'
                            f'border-radius:8px">{"".join(rows)}</div>')
        self.status.value = f"<b>guess {len(self.guesses)}/6</b>"

    def do_guess(self, _):
        wd = self.guess_box.value.strip().lower(); self.guess_box.value = ""
        if len(wd) != 5 or wd not in self.vocab.word_to_id:
            self.status.value = "not a valid word"; return
        fb = feedback(wd, self.secret)
        self.guesses.append(wd); self.feedbacks.append(fb)
        self.status.value = "solved!" if wd == self.secret else f"<b>guess {len(self.guesses)}/6</b>"
        self.render()

    def do_hint(self, _):
        if self.model is None:
            self.status.value = "load a checkpoint to get hints"; return
        from wordle02b.evaluate import next_guesses
        g = WordleGame(self.secret)
        g.guesses, g.feedbacks = list(self.guesses), list(self.feedbacks)
        self.status.value = f"hint: <b>{next_guesses(self.model, self.vocab, [g], device=device)[0].upper()}</b>"

    def do_autoplay(self, _):
        import time
        while self.secret not in self.guesses and len(self.guesses) < 6:
            g = WordleGame(self.secret)
            g.guesses, g.feedbacks = list(self.guesses), list(self.feedbacks)
            wd = next_guesses(self.model, self.vocab, [g], device=device)[0]
            fb = feedback(wd, self.secret)
            self.guesses.append(wd); self.feedbacks.append(fb)
            self.render(); time.sleep(0.5)
        self.status.value = ("solved!" if self.secret in self.guesses
                             else f"lost, word was {self.secret.upper()}")

ui = PlayUI(model, vocab)
display(w.VBox([ui.board, ui.guess_box, w.HBox([ui.go, ui.hint, ui.autoplay, ui.new]), ui.status]))

## What's next

- `notebooks/03_benchmark_api_models.ipynb` — pit this model against GPT, Claude, DeepSeek and friends
- `docs/improvement_walkthrough.md` — the playbook for pushing it past the teacher: self-play, RL, search, better teachers
- `app/gradio_app.py` — the same board as a standalone web app: `python app/gradio_app.py --checkpoint checkpoints/wordle-0.2b-final.pt`